In [1]:
!pip install accelerate==0.10.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.33.0
    Uninstalling accelerate-0.33.0:
      Successfully uninstalled accelerate-0.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.1 requires accelerate==0.33.0, but you have accelerate 0.10.0 which is incompatible.
pytdc 1.1.1 requires transformers==4.43.4, but you have transformers 4.20.1 which is incompatible.


In [2]:
!pip install -q wandb==0.15.12

In [3]:
cd ..

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
test_name = "i2b2"

In [5]:
# Dataset loading

In [6]:
from training.train_and_evaluate_relation_extraction import load_stored_dataset_combination_graph
from custom_datasets.dataframe_dataset import DFDataset

/workspace/llm-graph-construction


/workspace/llm-graph-construction/graph_building/llm/OpenChat.py:10: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama = Ollama(base_url='http://localhost:11434',


In [7]:
dataset_train, dataset_val, dataset_test_ub = load_stored_dataset_combination_graph(balanced=True, dataset=test_name)
number_of_relations = 3

In [8]:
transitivity = {
    ("OVERLAP", "OVERLAP"): "OVERLAP",
    ("BEFORE", "OVERLAP"): "BEFORE",
    ("BEFORE", "BEFORE"): "BEFORE",
    ("OVERLAP", "BEFORE"): "BEFORE",
    ("AFTER", "OVERLAP"): "AFTER",
    ("OVERLAP", "AFTER"): "AFTER",
    ("AFTER", "AFTER"): "AFTER",
    ("AFTER", "BEFORE"): "OVERLAP",
    ("BEFORE", "AFTER"): "OVERLAP"
}

inverse = {
    "BEFORE": "AFTER",
    "AFTER": "BEFORE",
    "OVERLAP": "OVERLAP"
}

In [9]:
def find_relation(graph, event1, event2, expected_relation):
    for ind, r in enumerate(graph):
        if r[0] == event1 and r[2] == event2 and r[1] == expected_relation:
            return ind, r
    for ind, r in enumerate(graph):
        if r[0] == event1 and r[2] == event2:
            return ind, r
    return -1, None
    
def does_relation_exist(graph, event1, event2):
    ind, relation = find_relation(graph, event1, event2, None)
    return ind >= 0

def add_transitive(graph):
    for i in range(len(graph)):
        for j in range(i+1, len(graph)):
            if graph[i][2] == graph[j][0]:
                # matching relations
                event1 = graph[i][0]
                event2 = graph[j][2]
                if not does_relation_exist(graph, event1, event2):
                    relation = transitivity[(graph[i][1], graph[j][1])]
                    new_relation = (event1, relation, event2)
                    graph.append(new_relation)
    return graph

def add_inverse(graph):
    for i in range(len(graph)):
        event1 = graph[i][2]
        event2 = graph[i][0]
        if not does_relation_exist(graph, event1, event2):
            relation = inverse[graph[i][1]]
            new_relation = (event1, relation, event2)
            graph.append(new_relation)
    return graph

def graph_closure(graph):
    graph = add_inverse(graph)
    graph = add_transitive(graph)
    return graph

# split into documents

In [10]:
dataframe = dataset_train.df
documents = set(dataframe["document_id"])
train_documents = []
for doc in documents:
    train_documents.append(dataframe[dataframe["document_id"] == doc].reset_index(drop=True))

# convert to graphs

In [11]:
def convert_to_graph(df):
    graph = []
    text = ""
    events = set()
    for row in df.iloc:
        text = row["text"]
        graph.append(((row["event1_start"], row["event1_end"], row["event1_text"]), row["class"], (row["event2_start"], row["event2_end"], row["event2_text"])))
        events.add((row["event1_start"], row["event1_end"], row["event1_text"]))
        events.add((row["event2_start"], row["event2_end"], row["event2_text"]))
    return graph, text, events

In [12]:
train_graphs = []
for i in range(len(train_documents)):
    train_graphs.append(convert_to_graph(train_documents[i]))

# compute closures

In [13]:
for i in range(len(train_documents)):
    train_graphs[i] = (graph_closure(train_graphs[i][0]), train_graphs[i][1], train_graphs[i][2])

# prepare training set

In [14]:
from spacy.lang.en import English
from spacy.tokenizer import Tokenizer
nlp = English()
tokenizer = Tokenizer(nlp.vocab)
def get_token_for_char(tokens, char_idx):
    for i, token in enumerate(tokens):
        if char_idx > token.idx:
            continue
        if char_idx == token.idx:
            return i, token
        if char_idx < token.idx:
            return i - 1, tokens[i - 1]
    return len(tokens) - 1, tokens[len(tokens) - 1]
def window_row_for_bert(text, e1, e2, window_size=60, add_event_markers=True):
    tokens = tokenizer(text)
    start = min(e1[0], e2[0])
    end = max(e1[1], e2[1])
    start_token, _ = get_token_for_char(tokens, start)
    end_token, _ = get_token_for_char(tokens, end)
    if end_token - start_token > window_size:
        return None
    start_token -= (window_size - (end_token - start_token)) // 2
    end_token += (window_size - (end_token - start_token)) // 2
    end_token += max(0, -start_token)
    start_token = max(0, start_token)
    end_token = min(end_token, len(tokens) - 1)
    start = tokens[start_token].idx
    end = tokens[end_token].idx + len(tokens[end_token])
    text = text[start:end]
    e1s = e1[0] - start
    e1e = e1[1] - start
    e2s = e2[0] - start
    e2e = e2[1] - start
    if add_event_markers:
        text = text[:e1s] + "<e1>" + text[e1s:e1e] + "</e1>" + text[e1e:]
        if e2s > e1s:
            e2s += 4
        if e2e > e1s:
            e2e += 4
        if e2s > e1e:
            e2s += 5
        if e2e > e1e:
            e2e += 5
        e1s += 4
        e1e += 4

        text = text[:e2s] + "<e2>" + text[e2s:e2e] + "</e2>" + text[e2e:]
        if e1s > e2s:
            e1s += 4
        if e1e > e2s:
            e1e += 4
        if e1s > e2e:
            e1s += 5
        if e1e > e2e:
            e1e += 5
        e2s += 4
        e2e += 4
    e1 = (e1s, e1e, e1[2])
    e2 = (e2s, e2e, e2[2])
    return text, e1, e2

In [15]:
from random import sample
examples = []

for graph, text, events in train_graphs:
    # add positive examples
    for e1, r, e2 in graph:
        window = window_row_for_bert(text, e1, e2)
        if window == None:
            continue
        windowed_text, e1, e2 = window
        examples.append({"text": windowed_text, "label": True})
    # add random examples
    events = list(events)
    events1 = sample(events, min(50, len(events)))
    events2 = sample(events, min(50, len(events)))
    for i in range(len(events1)):
        for j in range(len(events2)):
            e1 = events1[i]
            e2 = events2[j]
            window = window_row_for_bert(text, e1, e2)
            if window == None:
                continue
            exists = does_relation_exist(graph, e1, e2)
            if exists:
                # skip positive examples, as they were added already
                continue
            windowed_text, e1, e2 = window
            examples.append({"text": windowed_text, "label": exists})

In [16]:
print("All examples:", len(examples))
print("Positive examples:", len([e for e in examples if e["label"] == True]))
print("Negative examples:", len([e for e in examples if e["label"] == False]))

All examples: 98556
Positive examples: 45175
Negative examples: 53381


# Train the model

In [17]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

In [18]:
class TextClassificationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        text = self.examples[idx]["text"]
        label = 1 if self.examples[idx]["label"] else 0
        encoding = self.tokenizer(text, return_tensors='pt', padding="max_length", max_length=512, pad_to_max_length = True, truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label)}

In [19]:
import random
import numpy as np
random.Random(42).shuffle(examples)
train, val, test = np.split(examples, [int(.6*len(examples)), int(.8*len(examples))])
train_dataset = TextClassificationDataset(train)
val_dataset = TextClassificationDataset(val)
test_dataset = TextClassificationDataset(test)

Downloading:   0%|          | 0.00/226k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [22]:
mini_dataset = TextClassificationDataset(train[:5])

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f4d10851300>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f4d0bb61a20, raw_cell="mini_dataset = TextClassificationDataset(train[:5].." store_history=True silent=False shell_futures=True cell_id=6557e03a-9e42-4c72-a2e9-4ddcac413bc3>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

loading file https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt from cache at /root/.cache/huggingface/transformers/45c3f7a79a80e1cf0a489e5c62b43f173c15db47864303a55d623bb3c96f72a5.d789d64ebfe299b0e416afc4a169632f903f693095b4629a7ea271d5a0cf2c99
loading file https://huggingface.co/bert-base-uncased/resolve/main/added_tokens.json from cache at None
loading file https://huggingface.co/bert-base-uncased/resolve/main/special_tokens_map.json from cache at None
loading file https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json from cache at /root/.cache/huggingface/transformers/c1d7f0a763fb63861cc08553866f1fc3e5a6f4f07621be277452d26d71303b7e.76ea01b4b85ac16e2cec55c398cba7a943d89ab21dfdd973f6630a152e4b9aed
loading configuration file https://huggingface.co/bert-base-uncased/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/3c61d016573b14f7f008c02c4e51a366c67ab274726fe2910691e2a761acf43e.37395cee442ab11005bcd270f3c34464dc1704b715b5d7

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f4d10851300>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f4d0bb602e0, execution_count=22 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f4d0bb61a20, raw_cell="mini_dataset = TextClassificationDataset(train[:5].." store_history=True silent=False shell_futures=True cell_id=6557e03a-9e42-4c72-a2e9-4ddcac413bc3> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [33]:
def compute_metrics(p):
    predictions, labels = p
    print(predictions)
    print(labels)
    pred = np.argmax(predictions, axis=1)
    print(pred)
    tp = 0
    tn = 0
    fp = 0
    fn = 0
    for p, l in zip(pred, labels):
        if l > 0:
            if p > 0:
                tp += 1
            else:
                fn += 1
        else:
            if p > 0:
                fp += 1
            else:
                tn += 1

    if tp + fp > 0:
        p = tp / (tp + fp)
    else:
        p = 0
    if tp + fn > 0:
        r = tp/(tp + fn)
    else:
        r = 0
    if p + r > 0:
        f1 = (2*p*r)/(p+r)
    else:
        f1 = 0
    metrics = {"precision": p, "recall": r, "f1": f1}
    print(metrics)
    return metrics

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f4d10851300>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f4d09107eb0, raw_cell="def compute_metrics(p):
    predictions, labels = .." store_history=True silent=False shell_futures=True cell_id=a2f3e2d2-6d0f-44a3-a0a7-b64af3c8103d>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f4d10851300>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f4d091079a0, execution_count=33 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f4d09107eb0, raw_cell="def compute_metrics(p):
    predictions, labels = .." store_history=True silent=False shell_futures=True cell_id=a2f3e2d2-6d0f-44a3-a0a7-b64af3c8103d> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [34]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = BertForSequenceClassification.from_pretrained("bert-base-uncased")
model.to(device)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    auto_find_batch_size=True,
    num_train_epochs=5,
    weight_decay=0.001,
    gradient_accumulation_steps=1,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    push_to_hub=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=mini_dataset,
    eval_dataset=mini_dataset,
    compute_metrics=compute_metrics
)
trainer.train()
torch.save(model, "relation-detection-model.pt")

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f4d10851300>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f4d11a1c3a0, raw_cell="device = "cuda:0" if torch.cuda.is_available() els.." store_history=True silent=False shell_futures=True cell_id=b07a525e-5dc8-4c9d-ade2-ddcf11b4f828>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

loading configuration file https://huggingface.co/bert-base-uncased/resolve/main/config.json from cache at /root/.cache/huggingface/transformers/3c61d016573b14f7f008c02c4e51a366c67ab274726fe2910691e2a761acf43e.37395cee442ab11005bcd270f3c34464dc1704b715b5d7d52b1a461abe3b9e4e
Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.20.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file https://huggingface.co/bert-base-uncased/resolve/main/pytorch_model.bin from cache 

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,6.948300,0.685195,0,0.000000,0
2,1.873200,0.707735,0,0.000000,0
3,2.520000,2.295596,0,0.000000,0
4,2.038000,1.255492,0,0.000000,0
5,1.567800,1.701897,0,0.000000,0


***** Running Evaluation *****
  Num examples = 5
  Batch size = 1


...

***** Running Evaluation *****
  Num examples = 5
  Batch size = 1


[[ 1.756142  -2.4620566]
 [ 1.756142  -2.4620566]
 [ 1.756142  -2.4620566]
 [ 1.756142  -2.4620566]
 [ 1.756142  -2.4620566]]
[1 0 1 0 0]
[0 0 0 0 0]
{'precision': 0, 'recall': 0.0, 'f1': 0}




Training completed. Do not forget to share your model on huggingface.co/models =)




Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f4d10851300>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f4d09087ac0, execution_count=34 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f4d11a1c3a0, raw_cell="device = "cuda:0" if torch.cuda.is_available() els.." store_history=True silent=False shell_futures=True cell_id=b07a525e-5dc8-4c9d-ade2-ddcf11b4f828> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [ ]:
# Evaluate the model

In [ ]:
import torch
model = torch.load("relation-detection-model.pt")

In [ ]:
from torch.utils.data import DataLoader
import random
import numpy as np
random.Random(42).shuffle(examples)
train, val, test = np.split(examples, [int(.6*len(examples)), int(.8*len(examples))])
test_dataset = TextClassificationDataset(test)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=True)
model.to("cpu")
tp = 0
tn = 0
fp = 0
fn = 0
for example in test_dataloader:
    # print(example)
    label = example["label"]
    predictions = model(example["input_ids"], example["attention_mask"]).logits
    for i in range(len(label)):
        if label[i] == 1:
            # print(predictions)
            if predictions[i][1] >= predictions[i][0]:
                tp += 1
            else:
                fn += 1
        else:
            # print(predictions)
            if predictions[i][1] >= predictions[i][0]:
                fp += 1
            else:
                tn += 1
        print(tp, tn, fp, fn)
print(tp, tn, fp, fn)